# Context for Think, Observe and the Rest

**Prerequisites:** `04_context_construction.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

**The runtime decides *when* a component runs. Context engineering decides *what it sees*. Chapter 2 has nowhere to say the second thing.**

So every component says it for itself, inside its own `run()`:

| Component | What it gives the model | Decided in |
|---|---|---|
| `Think` | the whole conversation, rebuilt in the API's native tool-call shape | `Think._build_messages()` |
| `Observe` | the whole conversation flattened to text, plus the raw tool result | `Observe.run()` |
| `Graph` | the whole conversation, flattened, plus the raw result | `Graph._phrase()` |
| `ReflectionThink`, `SelectThink` | the whole conversation, flattened | each one's `run()` |

Four context decisions, four places, all of them "everything". None of them visible from the loop where the components are declared:

```python
Runtime(loop=[Think(tools=tools), Act(functions=functions), Observe()], repeat_from=0)
```

Nothing there says what any of them gets. This notebook makes that a thing you write, and it changes no chapter 2 code to do it. Two components, two very different answers: `Think` gets a scored selection of the run, and `Observe` gets only the pass it is reporting on.

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "04_memory"):
    sys.path.append(str(HERE.parent / chapter))

import observe as observe_module
from act import Act
from chat import Conversation, count_tokens
from context import available, tokens
from focus import Focus, View, current_pass, sizes, to_messages
from incident import RULES, incident
from observe import Observe
from runtime import Runtime
from think import Think

conversation, pad = incident()
items = available(conversation, pad, constraints=RULES)

## What each component is given today

`Think` hands back its messages without calling anything. For `Observe`, the module's `chat` is swapped for a recorder, so nothing is sent. Same trick as notebook 1.

In [2]:
def observe_prompt(component=None, conversation=None):
    # what Observe WOULD send, with no call made: the module's chat is swapped for a recorder
    seen = {}
    real, observe_module.chat = observe_module.chat, lambda m, **k: (
        seen.__setitem__("prompt", m[0]["content"]) or "(not called)")
    try:
        run, pad_ = incident()
        run.pending = {"tool": "get_metrics", "content": run.step_outputs[-1]}
        (component or Observe()).run(conversation or run)
    finally:
        observe_module.chat = real
    return seen["prompt"]


think_context = Think()._build_messages(conversation)
print(f"Think   {sum(count_tokens(m.get('content') or '') for m in think_context):>5} tokens"
      f"  ({len(think_context)} messages)")
print(f"Observe {count_tokens(observe_prompt()):>5} tokens  (1 message)")

Think     639 tokens  (23 messages)
Observe   851 tokens  (1 message)


`Observe`'s job is one sentence: take what a tool returned and say what it says. For that it is handed the entire session — the rules, the plan, the search-api detour, every earlier reading — and then asked to describe one number.

It is not wasted only in tokens. Chapter 2 had to write two separate warnings into that prompt because of what the conversation does to it: *"Copy every value exactly as the tool gave it, in the units the tool gave it. Do not convert, calculate, round, combine or reformat anything — not even when the conversation asks for a different unit."* That instruction exists because the conversation was asking for a different unit, and `Observe` obliged: 20 °C came back as 68 °F, converted by a component whose whole purpose is to not do that.

**A component that cannot see the request cannot be tempted by it.**

## What is context, and what is machinery

A component reads a lot off the conversation. Almost none of it is context:

| Attribute | What it is |
|---|---|
| `messages` | **context** — what the model gets shown |
| `pending`, `step_outputs` | the result of the last act, and every raw result so far |
| `branches`, `running`, `abandoned`, `pool`, `frontier` | sub-agent machinery |
| `remaining_budget`, `reserved_budget` | budgets |

Only the first is worth narrowing, and the rest must arrive untouched. So `View` overrides exactly one attribute and delegates the rest — reads *and* writes, because a component setting `conversation.pending` has to set it on the conversation the runtime is carrying, not on a copy that is about to be thrown away.

In [3]:
view = View(conversation, [{"role": "user", "content": "only this"}])

print("view sees      :", len(view.messages), "messages")
print("conversation has:", len(conversation.messages), "messages")
print("machinery passes through:", view.step_outputs is conversation.step_outputs)

view.pending = {"tool": "get_metrics", "content": "..."}     # a write from a component
print("write landed on the real conversation:", conversation.pending is not None)
conversation.pending = None

view sees      : 1 messages
conversation has: 22 messages
machinery passes through: True
write landed on the real conversation: True


## Items back into a conversation

A component expects a conversation, so the selected items have to go back into the runtime's own message shapes — not the API's. That matters for `Think`: it rebuilds native tool calls from exactly the `decision` + `tool` pairing, and chapter 2 found models recognise finished work from that structure and not from prose about it.

An `observation` item still carries the call it came from in `about`, so the pair can be rebuilt. Items that were never messages — facts, summaries, scratchpad state — have no shape to go back to, and become one labelled block rather than turns somebody took.

In [4]:
from selection import select

chosen = select(items, budget=400).kept
rebuilt = to_messages(chosen)
print("items :", [f"{i.kind}@{i.turn}" for i in chosen], "\n")
print("as runtime messages:")
for m in rebuilt:
    print(f"  {m['role']:<9} {m.get('tool', ''):<12} {' '.join((m.get('content') or '').split())[:60]}")

native = Think()._build_messages(View(conversation, rebuilt))
print("\nnative tool calls rebuilt by Think:",
      sum(1 for m in native if m.get("tool_calls")))

items : ['instruction@0', 'constraint@0', 'constraint@0', 'observation@6', 'observation@8', 'state@15', 'observation@19', 'tool_result@19', 'message@20', 'request@21', 'state@22', 'state@22'] 

as runtime messages:
  system                 You are an on-call assistant helping investigate production 
  system                 Everything you do must be read-only: do not restart, scale o
  system                 Give every duration in milliseconds.
  decision  get_logs     {"service": "checkout-api", "since": "07:00"}
  tool                   The checkout-api logs since 07:00 show repeated 'connection 
  decision  get_deploys  {"service": "checkout-api"}
  tool                   The last checkout-api deploy was v4.12.0 at 07:05, which cha
  plan                   Replanning: the database has to be ruled out before blaming 
  decision  get_metrics  {"service": "checkout-api"}
  tool                   checkout-api p95 latency is now 2.9 s.
  assistant              orders-db is healthy (query

## `Focus`: a component and the context it gets

`Focus` wraps a component. It works out which items that component should get, rebuilds them into a `View`, and calls the component with that. The component is unchanged and unaware; `Runtime` is unchanged and unaware, because a `Focus` is itself just something with `.run(conversation)`.

For `Observe` the right policy is not a scored selection at all. `Observe` phrases the result of the call that just happened, and that result arrives in `conversation.pending`, never through `messages`. So what it needs from the conversation is **this pass of the loop**: the standing request, so it names the entity correctly, and anything that came back since the last decision — which in a live loop is nothing, because `Observe` runs immediately after `Act`.

`current_pass` is that boundary, and it drops two things on purpose:

- **the rules, even though they are pinned.** Pinned means "must reach the component that answers the user", not "every component". Notebook 2 pinned them so nothing could lose them; here, losing them is the goal.
- **everything older** — the plan, the agent's own past answers, earlier readings.
- **the raw `tool_result` item**, because that same JSON already reaches `Observe` through `pending`. Keeping the item too put it in the prompt twice: 146 tokens of pure duplication on this session's metrics call.

What it does keep, besides the request, is the **decision itself** — the call `Think` just made, arguments included. That needed a change to `available()`: a `tool_call` decision is normally fused with the tool message after it into one `observation`, and mid-loop that message does not exist yet, because it is what `Observe` is about to produce. So the call in flight was in no item at all, and the one component whose job is to describe it could not see which call it was. It now arrives as a `decision` item.

A scored policy cannot say any of this. Given weights it will always trade part of this pass for something older that scored well.

In [5]:
watcher = Focus(Observe(), prepare=current_pass)

full = count_tokens(observe_prompt())
narrow = count_tokens(observe_prompt(watcher))
print(f"Observe, whole conversation: {full} tokens")
print(f"Observe, focused          : {narrow} tokens")
print("given:", [f"{i.kind}@{i.turn}" for i in watcher.last])

Observe, whole conversation: 851 tokens
Observe, focused          : 398 tokens
given: ['observation@19', 'request@21']


Two items, and neither is a rule: 851 tokens down to 398. It did not fall further because `Observe`'s message is three things: its own instructions (~180 tokens, fixed), the raw tool result (~150, which no context policy touches — trimming that is `invoke(max_tokens=)` in chapter 3 or `truncate()` in notebook 3), and the conversation. Only the third is ours to cut, and it went from ~660 tokens to ~40.

Note what `current_pass` keeps here: the observation from turn 19, because this canned session carries on for two more turns after that decision. Mid-loop — which is when `Observe` actually runs — the shape is different, and simpler:

Does it still do its job?

In [6]:
# the session cut off exactly where Observe runs: Act has carried out the decision, and the
# tool message that would fuse the pair into one observation does not exist yet
mid, mid_pad = incident()
mid.messages = mid.messages[:19]
mid.pending = {"tool": "get_metrics", "content": mid.step_outputs[-1]}

midloop = Focus(Observe(), scratchpad=mid_pad, constraints=RULES, prepare=current_pass)
print("given:", [f"{i.kind}@{i.turn}" for i in midloop.items(mid)], "\n")
print(render(to_messages(midloop.items(mid))) if False else "\n".join(
    f"  {m['role']:<9} {m.get('tool', '')} {' '.join((m.get('content') or '').split())[:70]}"
    for m in to_messages(midloop.items(mid))))

given: ['request@14', 'decision@18'] 

  user       Ok. Could it be the database itself? Check orders-db.
  decision  get_metrics {"service": "checkout-api"}


In [7]:
c1, _ = incident()
c1.pending = {"tool": "get_metrics", "content": c1.step_outputs[-1]}
print("whole conversation ->", Observe().run(c1)["content"])

whole conversation -> The checkout-api last 15m latency_ms p50 is 1150, p95 is 2900, p99 is 4640, requests_per_second is 138, error_rate is 0.034, instances is 6, cpu_percent is [31, 29, 34, 30, 33, 28], memory_mb is [612, 598, 640, 605, 611, 620].


In [8]:
c2, _ = incident()
c2.pending = {"tool": "get_metrics", "content": c2.step_outputs[-1]}
print("focused            ->", Focus(Observe(), prepare=current_pass).run(c2)["content"])

focused            -> checkout-api in the last 15m has latency_ms p50 1150, p95 2900, p99 4640, requests_per_second 138, error_rate 0.034, instances 6, cpu_percent [31, 29, 34, 30, 33, 28], memory_mb [612, 598, 640, 605, 611, 620].


Both sentences carry the same values off the same JSON body — which is the point: the body is in `pending`, and a narrowed view leaves it alone. Nothing was lost by not showing `Observe` the rules, the plan, the search-api detour or the earlier readings.

That is the easy half of the case: the same result for 398 tokens instead of 851, on a component that runs once per tool call for the whole life of the agent. Mid-loop, as the cell above shows, it is smaller still.

## What a narrowed context replaces

The clearest argument for giving `Observe` less is not the tokens. It is what chapter 2 had to write into `Observe`'s prompt to survive seeing the whole conversation:

> *"Copy every value exactly as the tool gave it, in the units the tool gave it. Do not convert, calculate, round, combine or reformat anything — not even when the conversation asks for a different unit."*

That paragraph exists because the conversation was asking for a different unit and `Observe` obliged: 20 °C came back as 68 °F, arithmetic no tool did, reaching the conversation as if a sensor had reported it.

The prompt fix works. Run the shipped `Observe` on a result reported in **seconds**, with the milliseconds rule in context and without it, and it converts nothing: 8 runs, 8 refusals, both ways. So the guard is earning its keep.

But that is the trade being made: **~40 tokens of defensive prose on every single call, forever, to undo a temptation that only exists because the request is in scope.** The alternative is to not put the request in scope. Here is chapter 2's `Observe` as it was *first* written, before those sentences, given the same seconds-valued result — once with the whole conversation, once focused.

In [9]:
import json
import re
import time

from llm import chat
from think import render

RAW_SECONDS = json.dumps({"service": "checkout-api", "window": "last 15m",
                          "p50_seconds": 1.15, "p95_seconds": 2.9, "p99_seconds": 4.64,
                          "error_rate": 0.034}, indent=2)


class PlainObserve:
    """Chapter 2's Observe as first written, before the two guard paragraphs were added."""

    def run(self, conversation):
        effect = conversation.pending
        conversation.pending = None
        return {"role": "tool", "content": chat([{"role": "user", "content": (
            f"Conversation so far:\n{render(conversation)}\n\n"
            f"The tool `{effect['tool']}` was just called and returned:\n{effect['content']}\n\n"
            "State this result as one plain factual sentence, using the conversation only to "
            "phrase it correctly (e.g. which units, which entity). No preamble.")}])}


def phrased(focused, trials=4):
    said = []
    for _ in range(trials):
        run, pad_ = incident()
        run.pending = {"tool": "get_metrics", "content": RAW_SECONDS}
        component = PlainObserve()
        said.append((Focus(component, prepare=current_pass).run(run)
                     if focused else component.run(run))["content"])
        time.sleep(16)                                  # the free tier's per-minute cap
    return said


for label, focused in (("whole conversation", False), ("focused", True)):
    group = phrased(focused)
    converted = sum(1 for s in group if re.search(r"\bms\b|millisecond", s))
    print(f"{label:<20} converted seconds to ms in {converted}/{len(group)}")
    for s in group:
        print("   ", " ".join(s.split())[:130])

whole conversation   converted seconds to ms in 4/4
    In the last 15 minutes, checkout‑api had a p50 latency of 1150 ms, a p95 latency of 2900 ms, a p99 latency of 4640 ms, and an erro
    In the last 15 minutes, checkout‑api’s p50 latency was 1150 ms, p95 latency 2900 ms, p99 latency 4640 ms, and its error rate was 0
    In the last 15 minutes, checkout‑api had a p50 latency of 1150 ms, a p95 latency of 2900 ms, a p99 latency of 4640 ms, and an erro
    In the last 15 minutes, checkout‑api had a p50 latency of 1150 ms, a p95 latency of 2900 ms, a p99 latency of 4640 ms, and an erro


focused              converted seconds to ms in 0/4
    In the last 15 minutes, the checkout‑api showed a p50 latency of 1.15 seconds, a p95 latency of 2.9 seconds, a p99 latency of 4.64
    In the last 15 minutes, the checkout‑api had a p50 latency of 1.15 s, a p95 latency of 2.9 s, a p99 latency of 4.64 s, and an erro
    In the last 15 minutes, checkout‑api recorded a p50 latency of 1.15 s, a p95 latency of 2.9 s, a p99 latency of 4.64 s, and an err
    In the last 15 minutes, the checkout‑api recorded a p50 latency of 1.15 s, a p95 latency of 2.9 s, a p99 latency of 4.64 s, and an


**4 of 4 against 0 of 4.** Fisher's exact test, one-sided: **p = 0.014**.

With the whole conversation in scope, chapter 2's original prompt read the rule — *"give every duration in milliseconds"* — and did as it was told: `1.15` became `1150 ms`, `2.9` became `2900 ms`. The arithmetic is right and no tool performed it. The numbers now in the conversation came from the model, and the agent's `calculate` tool, the one thing that could have been checked, was never called. That is chapter 2's `Observe` bug exactly, reproduced on demand.

Focused, the same prompt kept `2.9 s`, every time. Not because it was told not to convert — it was told nothing of the kind — but because **it could not see the request that wanted milliseconds.**

So there are two ways to stop a component doing something it shouldn't:

| | Cost | What it is |
|---|---|---|
| defensive prose in the prompt | ~40 tokens on **every** call, forever | telling the model not to be tempted |
| narrowing the context | one selection, cheaper than the prose | not tempting it |

Chapter 2 could only reach for the first, because it had nowhere to say the second. Both are worth having — the shipped `Observe` resisted 8 out of 8 with its guards *and* the rules in view — but only one of them scales to every new temptation somebody adds to a conversation later.

**This is the sharpest form of the chapter's claim.** Context is not a budget problem. What a component is shown decides what it will do.

## A whole loop, one policy per component

The same session, run for real this time: two tools, a `Think` that decides, an `Act` that calls them, an `Observe` that phrases what came back. Each model-calling component gets its own context; `Act` gets none, because `Act` calls no model — there is nothing to give it.

In [10]:
import json

from pydantic import BaseModel

from incident import _deploys, _metrics   # the same canned data the session was built from


class Service(BaseModel):
    service: str


def get_metrics(service):
    return _metrics("checkout-api", 2900, 1150, 138, 0.034) if "checkout" in service else \
        _metrics(service, 180, 64, 95, 0.0)


def get_deploys(service):
    return _deploys()


functions = {"get_metrics": get_metrics, "get_deploys": get_deploys}
schemas = {"get_metrics": Service, "get_deploys": Service}
tools = [{"type": "function", "function": {
    "name": name, "description": desc,
    "parameters": {"type": "object", "properties": {
        "service": {"type": "string", "description": "Service name, e.g. checkout-api"}},
        "required": ["service"]}}}
    for name, desc in (("get_metrics", "Latency and error metrics for one service."),
                       ("get_deploys", "Recent deploys for one service."))]

In [11]:
thinking = Focus(Think(tools=tools, allow_fork=False), constraints=RULES, budget=700)
watching = Focus(Observe(), prepare=current_pass)

run = Conversation(system="You are an on-call assistant helping investigate production incidents.")
run.messages.append({"role": "user", "content":
                     "checkout-api is slow. Check its metrics and its recent deploys, then tell "
                     "me in two sentences what is wrong."})

Runtime(loop=[thinking, Act(functions=functions, schemas=schemas), watching],
        repeat_from=0, max_iterations=10).run(run)

print(run.messages[-1]["content"], "\n")
print("last context given to each component:", sizes([thinking, watching]))

The deployment at 07:05 reduced the orders‑db connection pool from 50 to 5, which is likely throttling database access and causing the observed latency spikes (p99 ≈ 4640 ms) and elevated error rate (3.4%). This resource bottleneck explains the slowdown despite normal CPU and memory usage. 

last context given to each component: {'Think': 544, 'Observe': 33}


A working agent: two tool calls, an answer that names the cause, the deploy and the numbers, and every model-calling component on its own policy. `Act` has none, because `Act` calls no model — the runtime's only component that needs no context at all.

The last line is what the loop declaration could never say before: **`Think` was given 544 tokens and `Observe` 33.** Unfocused, `Observe` got 851 for the same job.

And `02_agent_runtime` was not touched to do any of this. That was the real test of chapter 2's design: a component is anything with `.run(conversation)`, so something that narrows a conversation and forwards the call is also a component. Chapter 4 found the same thing about memory, where `Recall` slots in beside `Observe`. Two chapters now have needed a hook that already existed.

## What you built

`focus.py`:

- **`View`**: a conversation with different `messages` and the same everything else. Writes to anything but `messages` land on the real conversation, so `pending`, `step_outputs` and the budgets keep working.
- **`to_messages(items)`**: items back into the runtime's shapes, with the `decision` + `tool` pairing rebuilt so `Think` still produces native tool calls.
- **`Focus(component, **policy)`**: one component and the context it is given. `prepare(items, conversation)` is for what selection cannot express — a boundary, a compression, a retrieval — and it runs per call, so what a component gets is decided when it runs rather than when the loop is declared.
- **`current_pass`**: `Observe`'s policy, and a boundary rather than a score — the standing request and the decision in flight, and nothing else. It drops the pinned rules on purpose, which is what stops the unit conversion.
- **`sizes(focused)`**: what each component was last given, in tokens.

Two changes were needed outside `focus.py`, both in this chapter: `available()` now emits a `decision` item for a call nothing has reported on yet (without it, the component whose job is to describe that call could not see which call it was), and `select()` gained `keep_last=N`, because recency as a *weight* guarantees nothing — see notebook 2.

What this notebook measured, in one line: giving `Observe` less cost nothing (same values, 398 tokens against 851, and 33 in the live loop), and it prevented a failure chapter 2 could only prevent with prose — 4/4 conversions against 0/4, p = 0.014.

What is still missing is a place to say all of this at once. `thinking` and `watching` above are two policies written beside each other by hand, and the compression from notebook 3 and the retrieval from notebook 2 are not wired into either. Notebook 6 puts them together: select and retrieve running side by side, compression when something triggers it, construction at the end, and one policy per component.